# Quantitative Analysis — t-SNE · UMAP · CKA · Entropy · Intrinsic Dim

Reads pre-computed parquet files (`frame.parquet`) and produces:
- **Step 3** — t-SNE 2D paired-lines plots (per pair, per layer)
- **Step 3'** — UMAP 2D paired-lines plots (per pair, per layer)
- **Step 4a** — Inter-cloud distance curves across layers
- **Step 4b** — Paired distance boxplots + Wilcoxon test
- **Step 4c** — Activation entropy per modality per layer
- **Step 5** — CKA layer × layer matrix
- **Step 6** — Intrinsic dimensionality per layer *(optional)*

> **Note on proxies**: Steps 4a/4b use PCA-2 coordinates from the parquets (exact).
> Steps 4c, 5, 6 use PCA-2 as a proxy for the full activation space — results are
> indicative. For full fidelity, load raw `.npy` activations (see Step 5 notes).

## 0 · Configuration

In [ ]:
from pathlib import Path

RUN_ID  = "rq1_full_pipeline_hypersim_100_scenes_4000_samples_join_null_A-20260518-225144"
DATASET = "hypersim"
SUBDIR  = "embeddings_umap"

PAIRS = [
    "rgb-depth",
    "depth-normals",
    "rgb-normals",
]

REPO_ROOT    = Path("..").resolve()
OUT_ROOT     = REPO_ROOT / "outputs" / RUN_ID / DATASET / "analysis"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

LAYERS_FILTER = None     # None = all available
MAX_LINES     = 300      # max paired lines per plot
MAX_FRAMES    = 5_000    # subsample for speed
RANDOM_SEED   = 42
DPI           = 150

PAIR_LABELS = {
    "rgb-depth":       "RGB ↔ Depth",
    "depth-normals":   "Depth ↔ Normals",
    "rgb-normals":     "RGB ↔ Normals",
}
def pl(p): return PAIR_LABELS.get(p, p)

print(f"Output: {OUT_ROOT}")

## 1 · Imports

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import wilcoxon

MOD_PALETTE = {"rgb": "#E07B39", "depth": "#4A90D9", "normals": "#57A85A"}
def mod_color(m): return MOD_PALETTE.get(m, "#888888")

def category_palette(cats):
    cmap = plt.get_cmap("tab20")
    return {c: cmap(i/max(len(cats)-1,1)) for i,c in enumerate(sorted(cats))}

print("Imports OK")

## 2 · Load parquets

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)

frames_by_pair   = {}  # {pair: {layer: df}}
pca_vars_by_pair = {}  # {pair: {layer: ndarray|None}}

for pair in PAIRS:
    emb_dir = REPO_ROOT / "results" / "embeddings" / RUN_ID / SUBDIR / pair
    if not emb_dir.exists():
        print(f"  [SKIP] {pair}"); continue
    avail = sorted(p.name for p in emb_dir.iterdir()
                   if p.is_dir() and p.name.startswith("layer_"))
    layers_to_use = LAYERS_FILTER or avail
    frames_by_pair[pair]   = {}
    pca_vars_by_pair[pair] = {}
    for layer in layers_to_use:
        fp = emb_dir / layer / "frame.parquet"
        if not fp.exists(): continue
        df = pd.read_parquet(fp)
        if MAX_FRAMES and len(df) > MAX_FRAMES:
            idx = rng.choice(len(df), MAX_FRAMES, replace=False)
            df  = df.iloc[idx].reset_index(drop=True)
        frames_by_pair[pair][layer] = df
        vp = emb_dir / layer / "pca_variance.csv"
        pca_vars_by_pair[pair][layer] = (
            pd.read_csv(vp)["explained_variance_ratio"].values[:2]
            if vp.exists() else None
        )
    print(f"  {pair}: {len(frames_by_pair[pair])} layers")

PAIRS_LOADED = [p for p in PAIRS if frames_by_pair.get(p)]
LAYERS = sorted({l for fp in frames_by_pair.values() for l in fp})
print(f"\nPairs  : {PAIRS_LOADED}")
print(f"Layers : {LAYERS}")

## 2b · Shared helpers

In [ ]:
def align_signs(all_frames, layers, x="tsne_x", y="tsne_y"):
    """
    Align t-SNE / UMAP sign across layers to prevent axis-flip artefact.
    Uses layer_00 as reference; picks flip that keeps modality centroids
    on the same side across all layers.
    """
    ref_layer = layers[0]
    ref_df    = all_frames[ref_layer]
    dims = [c for c in [x, y] if c in ref_df.columns]
    if len(dims) < 2:
        return all_frames  # coords not present

    def centroids(df):
        return {m: df.loc[df["modality"]==m, dims].mean().values
                for m in df["modality"].unique()}

    ref_c   = centroids(ref_df)
    aligned = {layers[0]: ref_df.copy()}
    for layer in layers[1:]:
        df = all_frames[layer].copy()
        best_cost, best_signs = np.inf, (1, 1)
        for sx in [1, -1]:
            for sy in [1, -1]:
                cand = df.copy()
                cand[dims[0]] *= sx; cand[dims[1]] *= sy
                c = centroids(cand)
                cost = sum(np.sum((ref_c[m]-c[m])**2)
                           for m in ref_c if m in c)
                if cost < best_cost:
                    best_cost, best_signs = cost, (sx, sy)
        df[dims[0]] *= best_signs[0]
        df[dims[1]] *= best_signs[1]
        aligned[layer] = df
    return aligned


def paired_distances(frame_df, x="pca_x", y="pca_y"):
    """
    Euclidean distance in (x,y) space between paired modalities
    for the same sample_key.  Returns 1-D array of distances.
    """
    if x not in frame_df.columns or y not in frame_df.columns:
        return np.array([])
    dedup = frame_df.drop_duplicates(subset=["sample_key","modality"])
    mods  = sorted(dedup["modality"].unique())
    if len(mods) < 2:
        return np.array([])
    a = (dedup[dedup["modality"]==mods[0]]
         .set_index("sample_key")[[x,y]])
    b = (dedup[dedup["modality"]==mods[1]]
         .set_index("sample_key")[[x,y]])
    common = a.index.intersection(b.index)
    return np.linalg.norm(a.loc[common].values - b.loc[common].values, axis=1)

print("Helpers loaded.")

---
## Step 3 — t-SNE 2D Paired Lines

Lines connect the **same image** encoded under two different modalities.
**Short line → the two representations are close → convergence for that sample.**
Sign-aligned across layers to prevent axis-flip artefacts.

### 3a — t-SNE 2D paired lines — key layers (00, 05, 11)

In [ ]:
KEY_LAYERS = ["layer_00", "layer_05", "layer_11"]

for pair in PAIRS_LOADED:
    fp = frames_by_pair[pair]
    layers_ok = [l for l in KEY_LAYERS if l in fp
                 and "tsne_x" in fp[l].columns]
    if not layers_ok: print(f"[SKIP] {pair} — tsne_x missing"); continue

    # align signs using available layers
    aligned = align_signs(fp, LAYERS, x="tsne_x", y="tsne_y")

    fig, axes = plt.subplots(1, len(layers_ok), figsize=(len(layers_ok)*7, 6))
    if len(layers_ok) == 1: axes = [axes]

    for ax, layer in zip(axes, layers_ok):
        df   = aligned.get(layer, fp[layer]).drop_duplicates(
                   subset=["sample_key","modality"])
        pal  = {m: mod_color(m) for m in df["modality"].unique()}

        # scatter
        for mod in sorted(pal):
            m = df["modality"]==mod
            ax.scatter(df.loc[m,"tsne_x"], df.loc[m,"tsne_y"],
                       c=[pal[mod]], label=mod, alpha=0.65, s=12, zorder=2)

        # paired lines
        drawn = 0
        for sk in df["sample_key"].unique():
            if drawn >= MAX_LINES: break
            rows = df[df["sample_key"]==sk]
            if len(rows)==2 and rows["modality"].nunique()==2:
                pts = rows[["tsne_x","tsne_y"]].values
                ax.plot(pts[:,0], pts[:,1], "k-",
                        alpha=0.12, lw=0.7, zorder=1)
                drawn += 1

        ax.set_title(f"{layer}", fontsize=11)
        ax.legend(title="Modality", markerscale=2, fontsize=8)
        ax.axis("off")

    fig.suptitle(f"t-SNE 2D — paired lines | {pl(pair)} | {DATASET}\n"
                 "short lines = per-sample convergence", fontsize=11)
    plt.tight_layout()
    out = OUT_ROOT / pair / "tsne2d_paired" / "key_layers.png"
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, dpi=DPI, bbox_inches="tight")
    plt.show(); print(f"→ {out}")

### 3b — t-SNE 2D paired lines — all 12 layers (grid)

In [ ]:
for pair in PAIRS_LOADED:
    fp      = frames_by_pair[pair]
    aligned = align_signs(fp, LAYERS, x="tsne_x", y="tsne_y")
    layers_ok = [l for l in LAYERS if l in aligned
                 and "tsne_x" in aligned[l].columns]
    if not layers_ok: continue

    ncols = 4; nrows = int(np.ceil(len(layers_ok)/ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*4, nrows*4))
    axes = axes.flatten()

    for i, layer in enumerate(layers_ok):
        ax = axes[i]
        df = aligned[layer].drop_duplicates(subset=["sample_key","modality"])
        pal = {m: mod_color(m) for m in df["modality"].unique()}
        for mod in sorted(pal):
            m = df["modality"]==mod
            ax.scatter(df.loc[m,"tsne_x"], df.loc[m,"tsne_y"],
                       c=[pal[mod]], label=mod, alpha=0.5, s=6, zorder=2)
        drawn = 0
        for sk in df["sample_key"].unique():
            if drawn >= MAX_LINES: break
            rows = df[df["sample_key"]==sk]
            if len(rows)==2 and rows["modality"].nunique()==2:
                pts = rows[["tsne_x","tsne_y"]].values
                ax.plot(pts[:,0], pts[:,1], "k-", alpha=0.1, lw=0.5, zorder=1)
                drawn += 1
        ax.set_title(layer, fontsize=8); ax.axis("off")

    for j in range(i+1, len(axes)): axes[j].set_visible(False)
    all_mods = sorted({m for df in fp.values() for m in df["modality"].unique()})
    handles = [mpatches.Patch(color=mod_color(m), label=m) for m in all_mods]
    fig.legend(handles=handles, title="Modality", loc="lower right", fontsize=8)
    fig.suptitle(f"t-SNE 2D — paired lines (all layers) | {pl(pair)}", fontsize=11)
    plt.tight_layout()
    out = OUT_ROOT / pair / "tsne2d_paired" / "grid_all_layers.png"
    fig.savefig(out, dpi=DPI, bbox_inches="tight")
    plt.show(); print(f"→ {out}")

---
## Step 3' — UMAP 2D Paired Lines

Same as Step 3 but in UMAP space. UMAP distances between clusters
are more meaningful than t-SNE — shorter lines here carry stronger evidence
of representational convergence.

### 3'a — UMAP 2D paired lines — key layers

In [ ]:
for pair in PAIRS_LOADED:
    fp = frames_by_pair[pair]
    layers_ok = [l for l in KEY_LAYERS if l in fp
                 and "umap_x" in fp[l].columns]
    if not layers_ok: print(f"[SKIP] {pair} — umap_x missing"); continue

    fig, axes = plt.subplots(1, len(layers_ok), figsize=(len(layers_ok)*7, 6))
    if len(layers_ok) == 1: axes = [axes]

    for ax, layer in zip(axes, layers_ok):
        df  = fp[layer].drop_duplicates(subset=["sample_key","modality"])
        pal = {m: mod_color(m) for m in df["modality"].unique()}
        for mod in sorted(pal):
            m = df["modality"]==mod
            ax.scatter(df.loc[m,"umap_x"], df.loc[m,"umap_y"],
                       c=[pal[mod]], label=mod, alpha=0.65, s=12, zorder=2)
        drawn = 0
        for sk in df["sample_key"].unique():
            if drawn >= MAX_LINES: break
            rows = df[df["sample_key"]==sk]
            if len(rows)==2 and rows["modality"].nunique()==2:
                pts = rows[["umap_x","umap_y"]].values
                ax.plot(pts[:,0], pts[:,1], "k-", alpha=0.12, lw=0.7, zorder=1)
                drawn += 1
        ax.set_title(layer, fontsize=11)
        ax.legend(title="Modality", markerscale=2, fontsize=8)
        ax.axis("off")

    fig.suptitle(f"UMAP 2D — paired lines | {pl(pair)} | {DATASET}\n"
                 "short lines = per-sample convergence", fontsize=11)
    plt.tight_layout()
    out = OUT_ROOT / pair / "umap2d_paired" / "key_layers.png"
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, dpi=DPI, bbox_inches="tight")
    plt.show(); print(f"→ {out}")

### 3'b — UMAP 2D paired lines — all 12 layers (grid)

In [ ]:
for pair in PAIRS_LOADED:
    fp = frames_by_pair[pair]
    layers_ok = [l for l in LAYERS if l in fp
                 and "umap_x" in fp[l].columns]
    if not layers_ok: continue

    ncols = 4; nrows = int(np.ceil(len(layers_ok)/ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*4, nrows*4))
    axes = axes.flatten()

    for i, layer in enumerate(layers_ok):
        ax = axes[i]
        df = fp[layer].drop_duplicates(subset=["sample_key","modality"])
        pal = {m: mod_color(m) for m in df["modality"].unique()}
        for mod in sorted(pal):
            m = df["modality"]==mod
            ax.scatter(df.loc[m,"umap_x"], df.loc[m,"umap_y"],
                       c=[pal[mod]], label=mod, alpha=0.5, s=6, zorder=2)
        drawn = 0
        for sk in df["sample_key"].unique():
            if drawn >= MAX_LINES: break
            rows = df[df["sample_key"]==sk]
            if len(rows)==2 and rows["modality"].nunique()==2:
                pts = rows[["umap_x","umap_y"]].values
                ax.plot(pts[:,0], pts[:,1], "k-", alpha=0.1, lw=0.5, zorder=1)
                drawn += 1
        ax.set_title(layer, fontsize=8); ax.axis("off")

    for j in range(i+1, len(axes)): axes[j].set_visible(False)
    all_mods = sorted({m for df in fp.values() for m in df["modality"].unique()})
    handles = [mpatches.Patch(color=mod_color(m), label=m) for m in all_mods]
    fig.legend(handles=handles, title="Modality", loc="lower right", fontsize=8)
    fig.suptitle(f"UMAP 2D — paired lines (all layers) | {pl(pair)}", fontsize=11)
    plt.tight_layout()
    out = OUT_ROOT / pair / "umap2d_paired" / "grid_all_layers.png"
    fig.savefig(out, dpi=DPI, bbox_inches="tight")
    plt.show(); print(f"→ {out}")

---
## Step 4 — Quantitative Summary Curves

### Step 4a — Inter-cloud distance curve

Mean Euclidean distance in **PCA-2 space** between paired modalities (same image)
across layers 0–11. Decreasing curve = progressive convergence.

In [ ]:
fig, axes = plt.subplots(1, len(PAIRS_LOADED),
                         figsize=(6*len(PAIRS_LOADED), 4), sharey=False)
if len(PAIRS_LOADED) == 1: axes = [axes]

for ax, pair in zip(axes, PAIRS_LOADED):
    fp = frames_by_pair[pair]
    layers_ok = [l for l in LAYERS if l in fp]
    means, stds, x = [], [], []
    for li, layer in enumerate(layers_ok):
        d = paired_distances(fp[layer], x="pca_x", y="pca_y")
        if len(d) == 0: continue
        means.append(np.mean(d)); stds.append(np.std(d)); x.append(li)
    means, stds = np.array(means), np.array(stds)
    ax.plot(x, means, marker="o", color="#4A90D9", lw=2, ms=5)
    ax.fill_between(x, means-stds, means+stds, alpha=0.2, color="#4A90D9",
                    label="±1 std")
    ax.set_xticks(x)
    ax.set_xticklabels([layers_ok[i].replace("layer_","L") for i in x], fontsize=8)
    ax.set_xlabel("Layer"); ax.set_ylabel("Mean paired distance (PCA-2)")
    ax.set_title(pl(pair), fontsize=10); ax.grid(alpha=0.3); ax.legend(fontsize=8)

fig.suptitle(f"Inter-modality paired distance across layers — {DATASET}\n"
             "Decreasing = progressive convergence", fontsize=11)
plt.tight_layout()
out = OUT_ROOT / "intercloud_distance_all_pairs.png"
fig.savefig(out, dpi=DPI, bbox_inches="tight")
plt.show(); print(f"→ {out}")

#### 4a — Overlaid (all pairs on same axes)

In [ ]:
pair_colors = {"rgb-depth":"#4A90D9","depth-normals":"#E07B39","rgb-normals":"#57A85A"}

fig, ax = plt.subplots(figsize=(9, 4))
for pair in PAIRS_LOADED:
    fp = frames_by_pair[pair]
    layers_ok = [l for l in LAYERS if l in fp]
    means, x = [], []
    for li, layer in enumerate(layers_ok):
        d = paired_distances(fp[layer])
        if len(d): means.append(np.mean(d)); x.append(li)
    c = pair_colors.get(pair, "#888")
    ax.plot(x, means, marker="o", color=c, lw=2, ms=5, label=pl(pair))

ax.set_xticks(range(len(LAYERS)))
ax.set_xticklabels([l.replace("layer_","L") for l in LAYERS], fontsize=9)
ax.set_xlabel("Layer"); ax.set_ylabel("Mean paired distance (PCA-2)")
ax.set_title(f"Inter-modality distance — all pairs — {DATASET}\n"
             "Lower = modalities closer in representation space", fontsize=10)
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
out = OUT_ROOT / "intercloud_distance_overlaid.png"
fig.savefig(out, dpi=DPI, bbox_inches="tight")
plt.show(); print(f"→ {out}")

### Step 4b — Paired distance boxplot + Wilcoxon test

Distribution of per-sample paired distances per layer.
Wilcoxon signed-rank test: layer 0 vs layer 11 (two-sided).
**p < 0.05 = the reduction in distance is statistically significant.**

In [ ]:
for pair in PAIRS_LOADED:
    fp = frames_by_pair[pair]
    layers_ok = [l for l in LAYERS if l in fp]
    dists = {l: paired_distances(fp[l]) for l in layers_ok}
    dists = {l: d for l, d in dists.items() if len(d) > 0}
    if not dists: print(f"[SKIP] {pair}"); continue

    fig, ax = plt.subplots(figsize=(14, 5))
    data    = [dists[l] for l in layers_ok if l in dists]
    labels  = [l.replace("layer_","L") for l in layers_ok if l in dists]

    bp = ax.boxplot(data, patch_artist=True, showfliers=False,
                    medianprops=dict(color="black", lw=1.5))
    for patch in bp["boxes"]:
        patch.set_facecolor("#AEC6E8"); patch.set_alpha(0.75)

    ax.set_xticks(range(1, len(labels)+1))
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_xlabel("Layer"); ax.set_ylabel("Paired distance (PCA-2)")
    ax.set_title(f"Paired modality distance per layer\n{pl(pair)} | {DATASET}",
                 fontsize=10)
    ax.grid(axis="y", alpha=0.3)

    # Wilcoxon: first vs last layer
    keys = [l for l in layers_ok if l in dists]
    d0, dn = dists[keys[0]], dists[keys[-1]]
    n = min(len(d0), len(dn))
    try:
        _, p = wilcoxon(d0[:n], dn[:n], alternative="two-sided")
        stars = "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "n.s."
        p_str = f"p = {p:.2e}" if p >= 1e-4 else "p < 1e-4"
        ax.annotate(
            f"Wilcoxon {keys[0]} vs {keys[-1]}:  {p_str}  {stars}",
            xy=(0.5,0.97), xycoords="axes fraction",
            ha="center", va="top", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.85),
        )
    except Exception as e:
        print(f"  Wilcoxon failed: {e}")

    plt.tight_layout()
    out = OUT_ROOT / pair / "paired_distance_boxplot.png"
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, dpi=DPI, bbox_inches="tight")
    plt.show(); print(f"→ {out}")

### Step 4c — Activation entropy per modality per layer

**Shannon entropy** of the binned PCA-2 activation distribution per modality.
> Proxy: uses `pca_x/pca_y` coordinates. For true entropy, load raw activations.

**High entropy** = activations spread across many values (diffuse representation).
**Low entropy** = activations concentrated (compact representation).
Expected pattern: rise in early layers → fall in late layers ("hunchback").

In [ ]:
def compute_entropy(frame_df, n_bins=60):
    """Shannon entropy (bits) of the pca_x/pca_y distribution per modality."""
    results = {}
    if "pca_x" not in frame_df.columns: return results
    for mod in frame_df["modality"].unique():
        vals = frame_df.loc[frame_df["modality"]==mod, ["pca_x","pca_y"]].values.flatten()
        counts, _ = np.histogram(vals, bins=n_bins)
        counts = counts[counts > 0]
        probs  = counts / counts.sum()
        results[mod] = -np.sum(probs * np.log2(probs + 1e-12))
    return results


for pair in PAIRS_LOADED:
    fp = frames_by_pair[pair]
    layers_ok = [l for l in LAYERS if l in fp]

    # compute entropy per layer per modality
    ent = {}  # {mod: [entropy_per_layer]}
    for layer in layers_ok:
        layer_ent = compute_entropy(fp[layer])
        for mod, h in layer_ent.items():
            ent.setdefault(mod, []).append(h)

    fig, ax = plt.subplots(figsize=(9, 4))
    x = range(len(layers_ok))
    for mod, values in sorted(ent.items()):
        ax.plot(x, values, marker="o", ms=5, lw=2,
                color=mod_color(mod), label=mod)

    ax.set_xticks(x)
    ax.set_xticklabels([l.replace("layer_","L") for l in layers_ok], fontsize=9)
    ax.set_xlabel("Layer")
    ax.set_ylabel("Shannon entropy (bits)")
    ax.set_title(f"Activation entropy per modality\n{pl(pair)} | {DATASET}\n"
                 "(computed on PCA-2 proxy — higher = more diffuse)",
                 fontsize=10)
    ax.legend(title="Modality"); ax.grid(alpha=0.3)
    plt.tight_layout()
    out = OUT_ROOT / pair / "entropy_per_layer.png"
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, dpi=DPI, bbox_inches="tight")
    plt.show(); print(f"→ {out}")

#### 4c — Entropy convergence: gap between modalities across layers

In [ ]:
# Plot absolute entropy gap between the two modalities per layer
# Shrinking gap = the two modalities are being processed with similar complexity
fig, axes = plt.subplots(1, len(PAIRS_LOADED),
                         figsize=(6*len(PAIRS_LOADED), 4), sharey=False)
if len(PAIRS_LOADED) == 1: axes = [axes]

for ax, pair in zip(axes, PAIRS_LOADED):
    fp = frames_by_pair[pair]
    layers_ok = [l for l in LAYERS if l in fp]
    ent = {}
    for layer in layers_ok:
        for mod, h in compute_entropy(fp[layer]).items():
            ent.setdefault(mod, []).append(h)
    mods = sorted(ent)
    if len(mods) < 2: continue
    gaps = np.abs(np.array(ent[mods[0]]) - np.array(ent[mods[1]]))
    x = range(len(layers_ok))
    ax.plot(x, gaps, marker="o", color="#9B59B6", lw=2, ms=5)
    ax.fill_between(x, 0, gaps, alpha=0.15, color="#9B59B6")
    ax.set_xticks(x)
    ax.set_xticklabels([l.replace("layer_","L") for l in layers_ok], fontsize=8)
    ax.set_xlabel("Layer"); ax.set_ylabel("|H(mod1) − H(mod2)|  (bits)")
    ax.set_title(f"{pl(pair)}", fontsize=10)
    ax.grid(alpha=0.3)

fig.suptitle(f"Entropy gap between modalities across layers — {DATASET}\n"
             "Shrinking gap = modalities processed with similar complexity",
             fontsize=11)
plt.tight_layout()
out = OUT_ROOT / "entropy_gap_all_pairs.png"
fig.savefig(out, dpi=DPI, bbox_inches="tight")
plt.show(); print(f"→ {out}")

---
## Step 5 — CKA Layer × Layer Matrix

12×12 heatmap where entry (i, j) = CKA between layer i and layer j activations.

> **Note**: computed here on PCA-2 features (proxy).
> For full fidelity, replace `X` below with raw activations loaded from `.npy` files.

**What to look for:**
- High values near the diagonal → consecutive layers are similar (plateau)
- Block structure → phases of representation learning
- Low off-diagonal → early and late layers are fundamentally different

In [ ]:
def linear_cka(X, Y):
    """Centred Kernel Alignment between two (N × D) matrices."""
    def centre(K):
        n = K.shape[0]
        H = np.eye(n) - np.ones((n,n))/n
        return H @ K @ H
    Kx = centre(X @ X.T)
    Ky = centre(Y @ Y.T)
    num = np.linalg.norm(Kx * Ky, "fro")
    den = np.linalg.norm(Kx,"fro") * np.linalg.norm(Ky,"fro")
    return float(num / (den + 1e-12))


def compute_cka_matrix(frames, layers, modality, max_n=1500, seed=42):
    """
    Build a (n_layers × n_layers) CKA matrix for one modality.
    Uses pca_x/pca_y as proxy activations.
    """
    rng2 = np.random.default_rng(seed)
    acts = {}
    for layer in layers:
        if layer not in frames: continue
        df  = frames[layer]
        sub = df.loc[df["modality"]==modality, ["pca_x","pca_y"]].values
        if len(sub) > max_n:
            sub = sub[rng2.choice(len(sub), max_n, replace=False)]
        acts[layer] = sub.astype(np.float32)

    layers_ok = [l for l in layers if l in acts]
    n = len(layers_ok)
    mat = np.zeros((n, n))
    for i, li in enumerate(layers_ok):
        for j, lj in enumerate(layers_ok):
            if i <= j:
                v = linear_cka(acts[li], acts[lj])
                mat[i,j] = mat[j,i] = v
    return mat, layers_ok


for pair in PAIRS_LOADED:
    fp   = frames_by_pair[pair]
    mods = sorted({m for df in fp.values() for m in df["modality"].unique()})

    ncols = len(mods)
    fig, axes = plt.subplots(1, ncols, figsize=(6*ncols, 5))
    if ncols == 1: axes = [axes]

    for ax, mod in zip(axes, mods):
        mat, layers_ok = compute_cka_matrix(fp, LAYERS, mod)
        ticks = [l.replace("layer_","L") for l in layers_ok]
        im = ax.imshow(mat, vmin=0, vmax=1, cmap="viridis", aspect="auto")
        plt.colorbar(im, ax=ax, label="CKA")
        ax.set_xticks(range(len(ticks))); ax.set_xticklabels(ticks, fontsize=8, rotation=45)
        ax.set_yticks(range(len(ticks))); ax.set_yticklabels(ticks, fontsize=8)
        ax.set_title(f"modality: {mod}", fontsize=10)

    fig.suptitle(f"CKA layer × layer — {pl(pair)} | {DATASET}\n"
                 "(proxy: PCA-2 features — for full fidelity use raw activations)",
                 fontsize=10)
    plt.tight_layout()
    out = OUT_ROOT / pair / "cka_layer_matrix.png"
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, dpi=DPI, bbox_inches="tight")
    plt.show(); print(f"→ {out}")

#### 5b — Using raw activations (optional, higher fidelity)

To replace the PCA-2 proxy with actual activation vectors, load the `.npy` files:

```python
import pandas as pd, numpy as np
from pathlib import Path

ACTS_ROOT = REPO_ROOT / "results" / "runs" / RUN_ID / "activations"
idx = pd.read_csv(REPO_ROOT / "results" / "runs" / RUN_ID /
                  "artifacts" / f"activation_index_{pair}.csv")

# Load activations for one layer and one modality
layer, modality = "layer_05", "rgb"
rows = idx[(idx["layer"]==layer) & (idx["modality"]==modality)]
X = np.stack([np.load(REPO_ROOT / r) for r in rows["activation_path"]])
# Then pass X to linear_cka() instead of the PCA-2 proxy
```

---
## Step 6 — Intrinsic Dimensionality per Layer *(optional)*

TwoNN estimator of the true dimensionality of the activation manifold at each layer.
Requires `scikit-dimension`: `pip install scikit-dimension --break-system-packages`

**Expected pattern ("hunchback")**: ID rises in early layers (expanding representation)
then falls in late layers (compression toward task-relevant structure).

**Convergence signal**: RGB and depth ID curves meeting = encoder treats both modalities
with similar geometric complexity → evidence of unification.

> Same PCA-2 proxy caveat applies — for reliable ID, use PCA-50 or raw activations.

In [ ]:
try:
    from skdim.id import TwoNN
    _SKDIM = True
except ImportError:
    print("scikit-dimension not installed.")
    print("pip install scikit-dimension --break-system-packages")
    _SKDIM = False

if _SKDIM:
    def compute_id(frames, layers, modality, max_n=1000, seed=42):
        rng2 = np.random.default_rng(seed)
        ids, layers_ok = [], []
        for layer in layers:
            if layer not in frames: continue
            df  = frames[layer]
            sub = df.loc[df["modality"]==modality, ["pca_x","pca_y"]].values
            if len(sub) > max_n:
                sub = sub[rng2.choice(len(sub), max_n, replace=False)]
            try:
                id_est = TwoNN().fit(sub.astype(np.float32)).dimension_
            except Exception:
                id_est = np.nan
            ids.append(id_est); layers_ok.append(layer)
        return ids, layers_ok

    for pair in PAIRS_LOADED:
        fp   = frames_by_pair[pair]
        mods = sorted({m for df in fp.values() for m in df["modality"].unique()})

        fig, ax = plt.subplots(figsize=(9, 4))
        for mod in mods:
            ids, layers_ok = compute_id(fp, LAYERS, mod)
            x = range(len(layers_ok))
            ax.plot(x, ids, marker="o", ms=5, lw=2,
                    color=mod_color(mod), label=mod)

        ax.set_xticks(range(len(layers_ok)))
        ax.set_xticklabels([l.replace("layer_","L") for l in layers_ok], fontsize=9)
        ax.set_xlabel("Layer")
        ax.set_ylabel("Intrinsic dimensionality (TwoNN)")
        ax.set_title(f"Intrinsic dimensionality per layer\n{pl(pair)} | {DATASET}\n"
                     "(proxy: PCA-2 — use PCA-50 or raw activations for full fidelity)",
                     fontsize=10)
        ax.legend(title="Modality"); ax.grid(alpha=0.3)
        plt.tight_layout()
        out = OUT_ROOT / pair / "intrinsic_dim_per_layer.png"
        out.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(out, dpi=DPI, bbox_inches="tight")
        plt.show(); print(f"→ {out}")

---
## Summary — output file tree

In [ ]:
def print_tree(root, indent=0):
    if not root.exists(): print(f"{'  '*indent}(not created yet)"); return
    for f in sorted(root.iterdir(), key=lambda p:(p.is_file(),p.name)):
        pre = "  "*indent
        if f.is_dir():  print(f"{pre}📁 {f.name}/"); print_tree(f, indent+1)
        else:
            sz = f.stat().st_size/1024
            print(f"{pre}  {f.name}  ({sz:.0f} KB)")

print(f"=== {OUT_ROOT} ===")
print_tree(OUT_ROOT)